# Qwen3-8B QLoRA Fine-tuned Model - Inference Demo

This notebook demonstrates how to load and run inference with the fine-tuned
Qwen3-8B medical reasoning model.

The LoRA adapter is merged into the **full-precision (bf16) base model** for
best accuracy. A memory-efficient 4-bit quantized alternative is provided in
Section 10 for GPUs with less than 16GB VRAM.

**Training summary:**
- Base model: Qwen/Qwen3-8B (8.2B params)
- Fine-tuning: QLoRA (4-bit NF4 + LoRA rank 16)
- Dataset: medical-o1-reasoning-SFT (English, 18.7k samples)
- Training: 4x A10G GPUs with DeepSpeed ZeRO-2, early-stopped at step 1400
- Final eval loss: 1.232

**VRAM requirement:** ~16GB for bf16 (default). ~6GB for 4-bit quantized (Section 10).

## 1. Install Dependencies

Run this cell if you haven't installed the required packages yet.

In [ ]:
# Uncomment and run if needed
!pip install torch transformers accelerate peft bitsandbytes

## 2. Load the Fine-tuned Model (Full Precision)

Load the base model in bf16, apply the LoRA adapter, then merge the adapter
weights permanently into the base model. This gives the best inference accuracy.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-8B"
ADAPTER_PATH = "../outputs/final_model"  # or "../outputs/lora_adapter"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model in bf16 (full precision, NO quantization)
print("Loading base model in bf16 (full precision)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

if torch.cuda.is_available():
    print(f"Base model GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Apply the LoRA adapter and merge into base model
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Merging LoRA weights into base model...")
model = model.merge_and_unload()
model.eval()

print("Full-precision merged model ready!")
if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Define the Inference Function

In [ ]:
def ask_medical_question(
    question: str,
    enable_thinking: bool = True,
    max_new_tokens: int = 1024,
    temperature: float = 0.6,
    top_p: float = 0.95,
    top_k: int = 20,
):
    """
    Ask a medical question and get a response with reasoning.

    Args:
        question: The medical question
        enable_thinking: If True, model shows its reasoning in <think> tags
        max_new_tokens: Maximum length of the generated response
        temperature: Controls randomness (lower = more deterministic)
        top_p: Nucleus sampling threshold
        top_k: Top-k sampling threshold

    Returns:
        dict with 'thinking', 'response', and 'full_output' keys
    """
    system_prompt = (
        "You are a medical AI assistant trained to provide detailed reasoning "
        "for medical questions. Think through the problem step-by-step before "
        "providing your final answer."
    )

    if enable_thinking:
        prompt = (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{question}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    else:
        prompt = (
            f"<|im_start|>system\n{system_prompt} "
            f"Answer directly without showing your reasoning process.<|im_end|>\n"
            f"<|im_start|>user\n{question}/no_think<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    result = {"full_output": full_output, "thinking": None, "response": full_output}

    if "<think>" in full_output and "</think>" in full_output:
        think_start = full_output.find("<think>") + len("<think>")
        think_end = full_output.find("</think>")
        result["thinking"] = full_output[think_start:think_end].strip()
        result["response"] = full_output[think_end + len("</think>"):].strip()

    return result


def display_result(result):
    """Pretty-print the model's response."""
    if result["thinking"]:
        print("=" * 60)
        print("REASONING")
        print("=" * 60)
        print(result["thinking"])
    print()
    print("=" * 60)
    print("ANSWER")
    print("=" * 60)
    print(result["response"])

## 4. Example: Medical Question with Reasoning

In [ ]:
%%time
question = "What are the key differences between Type 1 and Type 2 diabetes?"

result = ask_medical_question(question, enable_thinking=True)
display_result(result)

## 5. Example: Clinical Diagnosis Scenario

In [ ]:
question = (
    "A 55-year-old male presents with sudden onset crushing chest pain "
    "radiating to the left arm, diaphoresis, and shortness of breath. "
    "ECG shows ST elevation in leads II, III, and aVF. "
    "What is the most likely diagnosis and immediate management?"
)

result = ask_medical_question(question, enable_thinking=True)
display_result(result)

## 6. Example: Direct Answer (No Thinking Mode)

In [ ]:
question = "What are the common symptoms of pneumonia?"

result = ask_medical_question(question, enable_thinking=False)
display_result(result)

## 7. Adjusting Generation Parameters

You can control the model's output style by tuning these parameters:

| Parameter | Effect | Recommended Range |
|-----------|--------|-------------------|
| `temperature` | Randomness (lower = more focused) | 0.3 - 0.8 |
| `top_p` | Nucleus sampling cutoff | 0.9 - 0.99 |
| `top_k` | Top-k sampling cutoff | 10 - 50 |
| `max_new_tokens` | Maximum response length | 256 - 2048 |

In [ ]:
# More deterministic / focused response
result = ask_medical_question(
    "What is the treatment for community-acquired pneumonia?",
    temperature=0.3,
    top_p=0.9,
    top_k=10,
)
display_result(result)

In [ ]:
# More creative / exploratory response
result = ask_medical_question(
    "What is the treatment for community-acquired pneumonia?",
    temperature=0.8,
    top_p=0.95,
    top_k=50,
)
display_result(result)

## 8. Save the Merged Model (Optional)

Save the merged full-precision model so you can load it directly in future
without needing the base model + adapter files separately.
The saved model is a standard Hugging Face model (~16GB on disk).

In [ ]:
MERGED_MODEL_PATH = "../outputs/merged_model"

# Uncomment the lines below to save the merged model
# print("Saving merged full-precision model...")
# model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
# tokenizer.save_pretrained(MERGED_MODEL_PATH)
# print(f"Merged model saved to {MERGED_MODEL_PATH}")
#
# # To load later without any adapter:
# # model = AutoModelForCausalLM.from_pretrained(
# #     MERGED_MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto"
# # )

## 9. Cleanup

Free GPU memory when done.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")